# Data exploration

## Dependencies import

In [7]:
import numpy as np
import pandas as pd

print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")

NumPy: 2.5.4
pandas: 3.1.1


## Initial Data Review

In [ ]:
# Since the dataset file is saved in the Strict Open XML Spreadsheet format, we
# have to use `calamine` to read it
df = pd.read_excel("dataset/dataset-uci.xlsx", engine="calamine")

# Shape of dataset; data types; feature overview; missing values; duplicates; etc.
rows = len(df)
print(f"Number of rows: {rows}")
#how many columns we have of data
columns = len(df.columns)
print(f"Number of columns: {columns}")
##Now the actual shape of the dataset + types
print(f"Dataset shape: {df.shape}")
print(f"Dataset datatypes:\n{df.dtypes}")

print(f"# null values for each feature:\n{df.isnull().sum()}")
print(f"# null values: {df.isnull().sum().sum()}")

print(f"# duplicated values: {df.duplicated().sum()}")


Number of rows: 319
Number of columns: 39
Dataset shape: (319, 39)
Dataset datatypes:
Gallstone Status                                    int64
Age                                                 int64
Gender                                              int64
Comorbidity                                         int64
Coronary Artery Disease (CAD)                       int64
Hypothyroidism                                      int64
Hyperlipidemia                                      int64
Diabetes Mellitus (DM)                              int64
Height                                              int64
Weight                                            float64
Body Mass Index (BMI)                             float64
Total Body Water (TBW)                            float64
Extracellular Water (ECW)                         float64
Intracellular Water (ICW)                         float64
Extracellular Fluid/Total Body Water (ECF/TBW)    float64
Total Body Fat Ratio (TBFR) (%)             

## Cleaning

In [ ]:
print(f"# null values for each feature:\n{df.isnull().sum()}")
print(f"# null values: {df.isnull().sum().sum()}")

print(f"# duplicated values: {df.duplicated().sum()}")

# null values for each feature:
Gallstone Status                                  0
Age                                               0
Gender                                            0
Comorbidity                                       0
Coronary Artery Disease (CAD)                     0
Hypothyroidism                                    0
Hyperlipidemia                                    0
Diabetes Mellitus (DM)                            0
Height                                            0
Weight                                            0
Body Mass Index (BMI)                             0
Total Body Water (TBW)                            0
Extracellular Water (ECW)                         0
Intracellular Water (ICW)                         0
Extracellular Fluid/Total Body Water (ECF/TBW)    0
Total Body Fat Ratio (TBFR) (%)                   0
Lean Mass (LM) (%)                                0
Body Protein Content (Protein) (%)                0
Visceral Fat Rating (VFR)       

## Visualization

In [ ]:
y = df["Gallstone Status"]
X = df.drop(columns="Gallstone Status")

#How many ppl have gallstones and how many do not
gallstone_counts = y.value_counts()[1]
print(f"Number of people with gallstones: {gallstone_counts}")
print(f"Number of people without gallstones: {len(X) - gallstone_counts}")

## Preprocessing

### Outliers handling

In [ ]:
# Outlier
v_max =  X.max()
v_mean = X.mean()
v_min =  X.min()
v_std =  X.std()

z = (X - v_mean) / v_std
outliers = z.abs() > 10

print(outliers.sum())               # number of outliers per column
result = pd.DataFrame({
    "value": X[outliers].stack().dropna(),
    "z": z[outliers].stack().dropna(),
})

feature = result.index.get_level_values(1)
result["mean"] = v_mean[feature].values
result["std"] = v_std[feature].values

result = result[["value", "mean", "std", "z"]].rename_axis(["Record", "Feature"])
print(result.round(2))

Age                                               0
Gender                                            0
Comorbidity                                       0
Coronary Artery Disease (CAD)                     0
Hypothyroidism                                    0
Hyperlipidemia                                    0
Diabetes Mellitus (DM)                            0
Height                                            0
Weight                                            0
Body Mass Index (BMI)                             0
Total Body Water (TBW)                            0
Extracellular Water (ECW)                         0
Intracellular Water (ICW)                         0
Extracellular Fluid/Total Body Water (ECF/TBW)    0
Total Body Fat Ratio (TBFR) (%)                   0
Lean Mass (LM) (%)                                0
Body Protein Content (Protein) (%)                0
Visceral Fat Rating (VFR)                         0
Bone Mass (BM)                                    0
Muscle Mass 

In [ ]:
X = X.drop(40)
y = y.drop(40)

In [ ]:
X.loc[239, "Obesity (%)"] = 19.54

In [ ]:
X = X.drop(318)
y = y.drop(318)

### Skewed features

In [ ]:
# Continuous features only (binary and ordinal ones have ≤ 5 distinct values)
cont = [c for c in X.columns if X[c].nunique() > 5]

skew = X[cont].skew()
skewed = skew[skew > 1].index.tolist()             # strongly right-skewed
skewed = [c for c in skewed if (X[c] >= 0).all()]  # log1p needs x > -1

skew_table = pd.DataFrame({
    "Skewness": skew[skewed],
    "After log": np.log1p(X[skewed]).skew(),
}).sort_values("Skewness", ascending=False).round(2)

print(skew_table)
print(skew_table.to_latex())

## Feature Selection

In [ ]:
# Highly correlated features
corr = X.corr().abs()
upper = corr.where(np.triu(
    np.ones(corr.shape, dtype=bool),
    k=1
))  # upper triangle only

pairs = upper.stack()
high = pairs[pairs > 0.9]
print(high.sort_values(key=np.abs, ascending=False))

In [ ]:
keep = []
for c in X.columns:
    if not (corr.loc[c, keep] > 0.9).any():
        keep.append(c)

dropped = [c for c in X.columns if c not in keep]
print(f"Dropped {len(dropped)}:", dropped)

X = X[keep]